# Lab 4: Comparative Evaluation of Optimized Models Based on Size, Accuracy, and Inference Time

### Objective:
To compare original, pruned, and quantized TensorFlow Lite models on the basis of:
- File Size
- Accuracy
- Inference Time (Latency)

### Pre-requisites:
- Completion of Labs 1, 2, and 3
- TensorFlow and NumPy installed
- Models: `mnist_model.tflite`, `mnist_model_quant.tflite`, `mnist_pruned_model`

In [14]:
# Step 1: Import Required Libraries
import tensorflow as tf
import numpy as np
import time
import os
from tensorflow import keras

In [15]:
# Step 2: Load MNIST Test Data
mnist = keras.datasets.mnist
(_, _), (x_test, y_test) = mnist.load_data()
x_test = x_test / 255.0
x_test = x_test.astype(np.float32)
x_test_sample = x_test[:100]
y_test_sample = y_test[:100]

In [16]:
# Step 3: Define Evaluation Function for TFLite Models
def evaluate_tflite_model(model_path):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_index = input_details[0]['index']
    output_index = output_details[0]['index']

    correct = 0
    total_time = 0

    for i in range(len(x_test_sample)):
        input_data = np.expand_dims(x_test_sample[i], axis=0).astype(np.float32)
        interpreter.set_tensor(input_index, input_data)
        start = time.time()
        interpreter.invoke()
        end = time.time()
        output = interpreter.get_tensor(output_index)
        pred = np.argmax(output)
        if pred == y_test_sample[i]:
            correct += 1
        total_time += (end - start)

    accuracy = correct / len(x_test_sample)
    avg_time = total_time / len(x_test_sample)
    size_kb = os.path.getsize(model_path) / 1024

    return accuracy, avg_time, size_kb

In [17]:
# Step 4: Evaluate All Models
tflite_model_metrics = evaluate_tflite_model("mnist_model.tflite")
quantized_model_metrics = evaluate_tflite_model("mnist_model_quant.tflite")
pruned_model_metrics = evaluate_tflite_model("mnist_pruned_model.tflite")

C:\Users\bhawa\AppData\Roaming\Python\Python311\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [18]:
# Step 5: Display Comparison Table
import pandas as pd

df = pd.DataFrame({
    'Model': ['TFLite Model', 'Quantized Model', 'Pruned Model'],
    'Accuracy (%)': [tflite_model_metrics[0]*100, quantized_model_metrics[0]*100, pruned_model_metrics[0]*100],
    'Inference Time (s)': [tflite_model_metrics[1], quantized_model_metrics[1], pruned_model_metrics[1]],
    'Model Size (KB)': [tflite_model_metrics[2], quantized_model_metrics[2], pruned_model_metrics[2]]
})

df.set_index('Model', inplace=True)
df.round(2)

,Accuracy (%),Inference Time (s),Model Size (KB)
Model,,,
TFLite Model,100.0,0.0,429.57
Quantized Model,100.0,0.0,113.91
Pruned Model,100.0,0.0,429.54
